In [0]:
%sql
--SHOW CATALOGS;

-- CREATE SCHEMA IF NOT EXISTS workspace.bronze;
-- CREATE SCHEMA IF NOT EXISTS workspace.silver;
-- CREATE SCHEMA IF NOT EXISTS workspace.gold;


In [0]:
import re

# 1. Define your path
volume_file_path = '/Volumes/workspace/default/vinitvolume/Clinical Data_Discovery_Cohort.csv'

# 2. Read the CSV file exactly as-is into Spark
df_kaggle_raw = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(volume_file_path)
)

# 3. Clean column names (replace spaces and special characters with underscores)
for col_name in df_kaggle_raw.columns:
    # Convert spaces/hyphens to underscores and remove punctuation
    clean_name = re.sub(r'[ ,;{}()\n\t=]', '_', col_name)
    df_kaggle_raw = df_kaggle_raw.withColumnRenamed(col_name, clean_name)

# 4. Save safely to your Bronze Schema as a Delta Table
df_kaggle_raw.write.format("delta").mode("append").saveAsTable("workspace.bronze.patient_raw_data")

print("Bronze table successfully populated from our uploaded Kaggle file containing patient clinical data!")

In [0]:
from pyspark.sql.functions import col, when, to_date, try_to_date, upper, trim

# 1. Read from your Bronze table
bronze_df = spark.read.table("workspace.bronze.patient_raw_data")

# 2. Clean, standardise, and apply healthcare data compliance rules
staging_df  = (bronze_df
    # Deduplicate strictly on the unique PatientID
    .dropDuplicates(["PatientID"])
    
    # Filter out records where PatientID is missing entirely
    .filter(col("PatientID").isNotNull())
    
    # Standardise text statuses (removes leading/trailing spaces and forces uppercase)
    .withColumn("Dead_or_Alive", upper(trim(col("Dead_or_Alive"))))
    .withColumn("sex", upper(trim(col("sex"))))
    .withColumn("race", upper(trim(col("race"))))
    .withColumn("Stage", upper(trim(col("Stage"))))
    
    # Enforce medical data integrity:
    # If a patient is flagged as ALIVE, their Date_of_Death must be nullified
    .withColumn("Date_of_Death", 
        when(col("Dead_or_Alive") == "ALIVE", None)
        .otherwise(col("Date_of_Death"))
    )
    
    # Trim accidental spaces first, then safely parse using try_to_date to prevent pipeline crashes
    .withColumn("Specimen_date", try_to_date(trim(col("Specimen_date")), "M/d/yyyy"))
    .withColumn("Date_of_Death", try_to_date(trim(col("Date_of_Death")), "M/d/yyyy"))
    .withColumn("Date_of_Last_Follow_Up", try_to_date(trim(col("Date_of_Last_Follow_Up")), "M/d/yyyy"))
)

# 3. Save the structured, high-quality data to your Silver Schema
# silver_df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.cleaned_patient_outcomes")
# print("Silver table successfully built! Handled leading whitespace and malformed dates safely.")

# Register it as a temporary view so we can query it using standard SQL below
staging_df.createOrReplaceTempView("vw_silver_staging")

print("Staging data cleaned and ready for SQL Merge!")


In [0]:
%sql
-- First, ensure the target table structure exists so we have something to merge into
CREATE TABLE IF NOT EXISTS workspace.silver.cleaned_patient_outcomes (
    PatientID INT,
    Specimen_date DATE,
    Dead_or_Alive STRING,
    Date_of_Death DATE,
    Date_of_Last_Follow_Up DATE,
    sex STRING,
    race STRING,
    Stage STRING,
    Event INT,
    Time INT
) USING DELTA;

-- Perform the idempotent Delta Lake SQL MERGE
MERGE INTO workspace.silver.cleaned_patient_outcomes AS target
USING vw_silver_staging AS source
ON target.PatientID = source.PatientID

-- If the patient already exists, update all their clinical details
WHEN MATCHED THEN
  UPDATE SET *

-- If it's a completely new patient log, insert a new record
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
%sql
-- ==========================================
-- 1. PREPARE THE GOLD TARGET TABLES STRUCTURES
-- ==========================================
-- ==========================================
-- 1.1 INITIALISE THE SCD TYPE 2 TABLE STRUCTURE
-- ==========================================
-- Run this once to build the table with audit columns
-- ==========================================
-- INITIALISE THE BASE TARGET STRUCTURE
-- ==========================================
CREATE TABLE IF NOT EXISTS workspace.gold.dim_patient (
    PatientID INT,
    Patient_Sex STRING,
    Patient_Race STRING,
    Valid_From DATE,
    Valid_To DATE,
    Is_Current BOOLEAN
) USING DELTA;

-- ==========================================
-- 1.2 EXPLICITLY EVOLVE THE SCHEMA (ACL COMPLIANT BYPASS)
-- ==========================================
-- Instead of relying on automatic merging, we manually append the structural fields
-- These are harmless and will execute seamlessly even if the columns already exist
-- ALTER TABLE workspace.gold.dim_patient ADD COLUMNS (
--     Valid_From DATE,
--     Valid_To DATE,
--     Is_Current BOOLEAN
-- );

-- ==========================================
-- 1.3 STAGE INCOMING CHANGES
-- ==========================================
-- We create a temporary view of updates coming from Silver.
-- To trigger a Type 2 split, we look at rows where data has changed.
-- We calculate our changes and save them into a temporary target view.
-- Wrapping it here ensures Spark parses the columns cleanly, even on an empty table!
-- This view identifies exactly what needs to be changed
CREATE OR REPLACE TEMPORARY VIEW vw_scd2_incoming_changes AS
SELECT 
    s.PatientID, 
    s.sex AS Patient_Sex, 
    s.race AS Patient_Race,
    -- If the patient exists in Gold but their sex or race doesn't match Silver, flag it
    CASE WHEN t.PatientID IS NOT NULL AND (COALESCE(t.Patient_Sex, '') <> s.sex OR COALESCE(t.Patient_Race, '') <> s.race) THEN TRUE 
         ELSE FALSE 
    END AS Is_Update_Split
FROM (
    SELECT DISTINCT PatientID, sex, race 
    FROM workspace.silver.cleaned_patient_outcomes
) s
LEFT JOIN workspace.gold.dim_patient t 
  ON s.PatientID = t.PatientID AND t.Is_Current = TRUE;

-- -- Row Payload A: Pristine insertions and fresh updates mapping paths
-- SELECT 
--     PatientID, Patient_Sex, Patient_Race, 
--     CAST(current_date() AS DATE) AS Valid_From, 
--     CAST('9999-12-31' AS DATE) AS Valid_To, 
--     TRUE AS Is_Current,
--     -- Using a uniquely combined composite key prevents ambiguous source row collision errors
--     CAST(PatientID AS STRING) AS Merge_Join_Key
-- FROM cte_staged_updates

-- UNION ALL

-- -- Row Payload B: Expiration triggers to close history paths
-- SELECT 
--     PatientID, CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS DATE), CAST(NULL AS DATE), CAST(NULL AS BOOLEAN),
--     -- Appending an explicit string keyword separates this execution instruction entirely
--     CONCAT(PatientID, '_EXPIRE') AS Merge_Join_Key
-- FROM cte_staged_updates
-- WHERE Is_Update_Split = TRUE;

-- ==========================================
-- 1.4 BOOTSTRAP INITIAL INSERTION (IF EMPTY)
-- ==========================================
-- If this is the absolute first pass and the table has 0 rows, 
-- we run a direct initial load to securely establish the schema fingerprint.
-- INSERT INTO workspace.gold.dim_patient
-- SELECT PatientID, Patient_Sex, Patient_Race, Valid_From, Valid_To, Is_Current
-- FROM vw_final_scd2_staging_records
-- WHERE Merge_Join_Key NOT LIKE '%_EXPIRE'
--   AND NOT EXISTS (SELECT 1 FROM workspace.gold.dim_patient);

CREATE TABLE IF NOT EXISTS workspace.gold.fact_patient_outcomes (
    PatientID INT,
    Specimen_date DATE,
    Disease_Stage STRING,
    Event_Occurred INT,
    Survival_Time_Days INT,
    Dead_or_Alive STRING,
    Date_of_Death DATE,
    Date_of_Last_Follow_Up DATE
) USING DELTA;

-- ==========================================
-- 2. MERGE INTO PATIENT DIMENSION
-- ==========================================
-- ==========================================
-- 2.1 EXECUTE THE SCD TYPE 2 MERGE LOOP WITH CTE
-- ==========================================
-- This structure gracefully closes out old histories and spawns fresh active ones
-- Now that the source structure is isolated, the merge runs flawlessly
-- MERGE INTO workspace.gold.dim_patient AS target
-- USING (
--     SELECT * FROM vw_final_scd2_staging_records
--     WHERE EXISTS (SELECT 1 FROM workspace.gold.dim_patient)
-- ) AS source
-- -- We append '_EXPIRE' to the check here so it matches the explicit row instruction cleanly
-- ON CONCAT(target.PatientID, '_EXPIRE') = source.Merge_Join_Key 

-- -- ACTION 1: Expire old active histories cleanly when an update split signal hits
-- WHEN MATCHED AND target.Is_Current = TRUE THEN
--   UPDATE SET 
--     target.Valid_To = current_date(),
--     target.Is_Current = FALSE

-- -- ACTION 2: Insert the pristine new row or subsequent update history
-- WHEN NOT MATCHED AND source.Merge_Join_Key NOT LIKE '%_EXPIRE' THEN
--   INSERT (PatientID, Patient_Sex, Patient_Race, Valid_From, Valid_To, Is_Current)
--   VALUES (source.PatientID, source.Patient_Sex, source.Patient_Race, source.Valid_From, source.Valid_To, source.Is_Current);
-- ==========================================
-- STEP 1: EXPIRE CHANGED HISTORICAL RECORDS (UPDATE)
-- ==========================================
-- This finds active records in Gold that have a different value in Silver and closes them out
UPDATE workspace.gold.dim_patient AS target
SET 
    target.Valid_To = current_date(),
    target.Is_Current = FALSE
WHERE 
    target.Is_Current = TRUE
    AND target.PatientID IN (
        SELECT PatientID 
        FROM vw_scd2_incoming_changes 
        WHERE Is_Update_Split = TRUE
    );


-- ==========================================
-- STEP 2: INSERT NEW AND UPDATED ACTIVE RECORDS (INSERT)
-- ==========================================
-- This inserts completely new patients OR the fresh updated records for existing patients
INSERT INTO workspace.gold.dim_patient (PatientID, Patient_Sex, Patient_Race, Valid_From, Valid_To, Is_Current)
SELECT 
    src.PatientID, 
    src.Patient_Sex, 
    src.Patient_Race, 
    CAST(current_date() AS DATE) AS Valid_From, 
    CAST('9999-12-31' AS DATE) AS Valid_To, 
    TRUE AS Is_Current
FROM 
    vw_scd2_incoming_changes src
WHERE 
    -- Condition A: It's an update split (we just closed the old row, now we insert the new one)
    src.Is_Update_Split = TRUE
    -- Condition B: It's a brand new patient that doesn't exist in Gold at all
    OR NOT EXISTS (
        SELECT 1 
        FROM workspace.gold.dim_patient tgt 
        WHERE tgt.PatientID = src.PatientID
            AND tgt.Is_Current = TRUE
    );

-- ==========================================
-- 3. MERGE INTO OUTCOMES FACT TABLE (EXPLICIT MAPPING)
-- ==========================================
MERGE INTO workspace.gold.fact_patient_outcomes AS target
USING workspace.silver.cleaned_patient_outcomes AS source
ON target.PatientID = source.PatientID
WHEN MATCHED THEN
  UPDATE SET 
    target.Specimen_date         = source.Specimen_date,
    target.Disease_Stage         = source.Stage,       -- Maps 'Stage' to 'Disease_Stage'
    target.Event_Occurred        = source.Event,       -- Maps 'Event' to 'Event_Occurred'
    target.Survival_Time_Days    = source.Time,        -- Maps 'Time' to 'Survival_Time_Days'
    target.Dead_or_Alive         = source.Dead_or_Alive,
    target.Date_of_Death         = source.Date_of_Death,
    target.Date_of_Last_Follow_Up = source.Date_of_Last_Follow_Up
WHEN NOT MATCHED THEN
  INSERT (
    PatientID, Specimen_date, Disease_Stage, Event_Occurred, 
    Survival_Time_Days, Dead_or_Alive, Date_of_Death, Date_of_Last_Follow_Up
  ) 
  VALUES (
    source.PatientID, source.Specimen_date, source.Stage, source.Event, 
    source.Time, source.Dead_or_Alive, source.Date_of_Death, source.Date_of_Last_Follow_Up
  );

-- ==========================================
-- 4. CREATE EXECUTIVES VIEW (AUTOMATICALLY REFRESHES)
-- ==========================================
CREATE OR REPLACE VIEW workspace.gold.vw_survival_metrics_summary AS
SELECT 
    p.Patient_Sex,
    f.Disease_Stage,
    COUNT(f.PatientID) AS Total_Patients,
    SUM(CASE WHEN f.Dead_or_Alive = 'DEAD' THEN 1 ELSE 0 END) AS Total_Deaths,
    ROUND(AVG(f.Survival_Time_Days), 1) AS Avg_Survival_Days
FROM 
    workspace.gold.fact_patient_outcomes f
JOIN 
    workspace.gold.dim_patient p ON f.PatientID = p.PatientID
GROUP BY 
    p.Patient_Sex, f.Disease_Stage;

In [0]:
# %sql
# SELECT * FROM workspace.gold.vw_survival_metrics_summary 
# ORDER BY Disease_Stage ASC, Total_Patients DESC;